# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [ ]:
### API key management

### Reminder: Place .env file inside the root of the project folder so when calling the below from inside the notebook it should find the .env fule and load it inside the notebook environment
### PLEASE ADD THIS `.env` FILE TO YOUR PROJECT'S `.gitignore` file before committing and pushing the changes to your remote repo, as it contains API Keys and Secrets in it

import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

import os

print("OPENAI_API_KEY" in os.environ)
print("LANGCHAIN_API_KEY" in os.environ)
print("TAVILY_API_KEY" in os.environ)


In [ ]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [ ]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Loan Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [ ]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/complaints.csv",
    metadata_columns=[
      "Date received", 
      "Product", 
      "Sub-product", 
      "Issue", 
      "Sub-issue", 
      "Consumer complaint narrative", 
      "Company public response", 
      "Company", 
      "State", 
      "ZIP code", 
      "Tags", 
      "Consumer consent provided?", 
      "Submitted via", 
      "Date sent to company", 
      "Company response to consumer", 
      "Timely response?", 
      "Consumer disputed?", 
      "Complaint ID"
    ]
)

loan_complaint_data = loader.load()

for doc in loan_complaint_data:
    doc.page_content = doc.metadata["Consumer complaint narrative"]

Let's look at an example document to see if everything worked as expected!

In [ ]:
loan_complaint_data[0]

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "LoanComplaints".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [ ]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    loan_complaint_data,
    embeddings,
    location=":memory:",
    collection_name="LoanComplaints"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [ ]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [ ]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [12]:
naive_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

"Based on the provided context, the most common issues with loans appear to be related to mismanagement and errors by loan servicers, including:\n\n- Errors in loan balances and interest calculations\n- Incorrect or outdated information on credit reports\n- Inability to properly apply payments, especially to principal\n- Unauthorized transfer or sale of loans without proper notification\n- Discrepancies and lack of transparency in loan account details\n- Problems with repayment plans, forbearance, or forgiveness processes\n- Mishandling of personal information and violation of privacy laws\n\nAmong these, issues with errors in loan balances, interest, and account information seem to be the most frequently reported and prominent problems in the complaints. These errors often lead to negative impacts on credit scores and financial hardship.\n\nIf you'd like a specific summary, the most common issue with loans based on this data is often related to errors and mismanagement by loan service

In [13]:
naive_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, yes, some complaints did not get handled in a timely manner. Specifically, there are complaints where the response was marked as "No" for timely response:\n\n- Row 441 (Complaint ID: 12709087): The consumer reported that despite being told their application would be expedited and they would receive communication within 15 days, they had not heard back and described the response as not timely.\n- Row 816 (Complaint ID: 12832400): The consumer followed up on a complaint about a process issue and stated it had been over 18 months with no resolution, indicating a failure to handle the complaint in a timely manner.\n\nAdditionally, multiple complaints detailed delays or lack of response over extended periods, which suggests that certain complaints were not handled in a timely manner.'

In [14]:
naive_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People often fail to pay back their loans due to several interconnected reasons highlighted in the complaints:\n\n1. **Interest Accumulation and Unmanageable Payments**: Many borrowers find that options like forbearance or deferment lead to ongoing interest accumulation, which increases the total amount owed over time. Lowering monthly payments just extends the repayment period and can make it impossible to catch up.\n\n2. **Lack of Clear Information and Communication**: Borrowers report being inadequately informed about their loan status, repayment requirements, or transfer details. For example, they were not notified when loans were transferred between servicers or when payments were resumed, leading to missed payments or delinquency.\n\n3. **Difficulty Accessing or Understanding Payment Plans**: Some complain that online systems are not user-friendly or are down, preventing borrowers from applying for income-driven repayment plans or making timely payments.\n\n4. **Incorrect or Con

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [15]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(loan_complaint_data, )

We'll construct the same chain - only changing the retriever.

In [16]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [17]:
bm25_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issue with loans appears to be problems related to handling and management of the loan by servicers, including disputes over fees, applying payments correctly, and providing accurate information about loan balances and terms. Specifically, issues such as "Dealing with your lender or servicer" is frequently mentioned, with sub-issues like disagreements over fees, trouble with payment application, and bad information about the loan.\n\nSo, the most common issue with loans, according to this data, is difficulties and disputes arising from loan servicers\' handling of loans, including lack of transparency, incorrect information, and improper application of payments.'

In [18]:
bm25_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, all the complaints listed were responded to with a notification of "Closed with explanation," and the responses are indicated as "Timely response? Yes" for each. This suggests that the complaints were handled within the expected timeframe. Therefore, there are no indications of complaints not being handled in a timely manner.'

In [19]:
bm25_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily because of issues such as being steered into incorrect payment plans or forbearances, lack of proper communication from loan servicers, and administrative errors. For example, some borrowers were misled into unsuitable forbearances that increased their principal balances, while others experienced repeated payment reversals due to system errors or incorrect information from the servicers. Additionally, there were cases where borrowers did not receive necessary notifications about changes to their accounts, leading to missed payments or negative credit impacts. Overall, failures to pay are often linked to administrative mistakes, deceptive practices, poor communication, and errors made by loan servicers or the agencies managing the loans.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [20]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [21]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [22]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issue with loans appears to be problems related to dealing with lenders or servicers, including errors, miscommunication, and mishandling of loan information. Specifically, issues such as errors in loan balances, misapplied payments, wrongful denials of payment plans, incorrect or incomplete information, and improper handling of loan data and transfers are frequently mentioned.'

In [23]:
contextual_compression_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, it appears that at least one complaint was handled in a timely manner, with the response marked as "Yes" for timely response. However, the complaints do mention delays and ongoing issues:\n\n- One complaint about delayed responses and unresolved issues has been open for over 1 year and nearly 18 months without resolution.\n- Another complaint about unaddressed main issues has been ongoing for over 2-3 weeks.\n  \nWhile some complaints were managed quickly, the ongoing nature of several reports indicates that not all complaints were handled in a timely manner. Therefore, yes, some complaints did not get handled in a timely manner.\n'

In [24]:
contextual_compression_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily due to a lack of clear communication, understanding, and support from loan servicers. Several factors contributed to this problem:\n\n1. **Lack of Awareness and Information:** Borrowers often did not know they had to repay their loans or were not properly informed about their loan obligations. For example, some borrowers were unaware that they had to make payments until years later, and did not receive detailed breakdowns of their loan balances or interest.\n\n2. **Poor Communication and Notification Failures:** Borrowers reported not being notified about payment due dates, loan transfers, or the need to start repayment. In some cases, they were not contacted when loans were bought by different servicers, leading to confusion and missed payments.\n\n3. **Interest Accumulation and Financial Hardship:** Many borrowers found it difficult to manage payments because of continued interest accumulation, especially when loans were put into forbe

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [25]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
)

In [26]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [27]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the context provided, appears to be misconduct and mismanagement by loan servicers, including errors in account handling, inaccurate information about balances and interest, improper transfers of loans without proper notification or consent, and difficulties in communication and documentation. Specifically, many complaints involve:\n\n- Errors in loan balances and interest calculations\n- Difficulties applying payments or paying down principal\n- Unnotified transfers or reassignments of loans\n- Unclear or misleading information about loan terms and repayment options\n- Problems with loan classification and in-school deferments\n- Attempts by servicers to enforce invalid or disputed debts\n- Poor customer service, including silent calls and lack of transparency\n\nOverall, the most common issue seems to be that borrowers face unfair, inaccurate, and poorly managed loan servicing practices, which can lead to unnecessary financial hardship, con

In [28]:
multi_query_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints data, several complaints indicate issues with timely handling or response. Specifically, some complaints mention that the organization failed to respond within the expected timeframe or ignored requests altogether. For example, the complaint with Complaint ID 13091395 notes that the company did not respond **as of XX/XX/year**, which was over a year after the initial complaint, indicating a lack of timely handling. Similarly, Complaint ID 12709087 states that the complaint was "No response" after a significant delay, and the complainant reports ongoing issues despite submitting requests over a year ago.\n\nAdditionally, multiple complaints mention that the organization failed to address issues promptly, and some complaints highlight that the complaint was not resolved after extended periods—over a year in some cases. There are also multiple references to complaints being "Closed with explanation" or "no response" despite ongoing issues, which suggests 

In [29]:
multi_query_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for several reasons, including:\n\n1. **Interest Accumulation and Compounding:** Borrowers often found that putting loans into forbearance or deferment allowed interest to continue accumulating and compounding, making the total repayment amount grow significantly and extending the payoff period. Lowering monthly payments could result in interest negating payments made, increasing total debt over time.\n\n2. **Lack of Adequate Information and Support:** Many borrowers were not fully informed about repayment options such as income-driven repayment plans or rehabilitation programs. Servicers sometimes steered borrowers into long-term forbearances without explaining the impact on interest and forgiveness opportunities.\n\n3. **Economic Hardship and Unaffordable Payments:** Borrowers faced financial hardships, unemployment, or stagnant wages, making it difficult or impossible to increase payments or qualify for forgiveness programs, leading to continue

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [30]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = loan_complaint_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [31]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [32]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [33]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [34]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [35]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the provided complaints, appears to be related to "Federal student loan servicing," with significant concerns around errors in loan balances, misapplied payments, wrongful denials of payment plans, and problems with credit reporting and information accuracy. Many complaints involve systemic issues such as incorrect reporting, unverified debts, unfair practices, and difficulties in managing or understanding loan terms.\n\nHowever, if I had to identify a broad pattern, issues with loan servicing—particularly errors and mismanagement by loan servicers—seem to be the most prevalent concern among these complaints.'

In [36]:
parent_document_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, yes, some complaints did not get handled in a timely manner. Specifically, two complaints filed with MOHELA on 03/28/25 and 04/11/25, both related to student loan servicing issues, were marked as "Timely response?": "No." This indicates that these complaints were not addressed promptly.'

In [37]:
parent_document_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for various reasons, including experiencing severe financial hardship, lack of proper information or guidance about repayment obligations, and issues with the loan servicing process. For example, some borrowers did not receive timely notifications or clear communication from loan servicers about when to start payments or changes in loan ownership, which led to missed payments and delinquencies. Others faced difficulties because their educational institutions closed or provided misleading information about job prospects and financial obligations, making it hard for graduates to repay their loans. Additionally, some borrowers relied on deferment or forbearance due to economic or health challenges, which increased the interest and further complicated repayment. Overall, inadequate communication, unexpected financial hardships, and institutional issues contributed to the failure to repay loans.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [38]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [39]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [40]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

"The most common issue with loans, based on the provided complaints, appears to be dealing with the handling and management of student loans by servicers. Specific recurring issues include:\n\n- Errors and bad information about loan balances, interest calculations, and account statuses.\n- Problems with repayment plans, including trouble with payments, finding suitable options, and tracking account activity.\n- Mismanagement during transfers between different servicers, resulting in unauthorized transfers, incorrect classifications (e.g., FFELP, HEAL loans), or ending deferments prematurely.\n- Lack of transparency, inadequate communication, and failure to provide required documentation such as Master Promissory Notes or repayment histories.\n- Improper reporting to credit bureaus, including incorrect delinquency statuses, late payments, or increased balances without explanation.\n- Inadequate customer service, including unresponsiveness, silence calls, and inability to get accurate in

In [41]:
ensemble_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints data, yes, there are multiple instances where complaints indicate that issues were not handled in a timely manner. Specific examples include:\n\n- Complaint ID 12709087 (row 441): The consumer states that the company’s response was "No" in terms of being timely, and notes ongoing unprocessed issues with their loan application, with no response from the company after several calls and extended wait times.\n- Complaint ID 12935889 (row 123): The complaint was marked "No" for timely response, indicating the company failed to respond within the expected timeframe.\n- Complaint ID 12973003 (row 66): The complaint was handled within the timeframe, but the consumer still indicates unresolved issues with delays.\n- Complaint ID 12964633 (row 95): The complaint was marked "Yes" for timely response, but the ongoing unresolved problems and delays in addressing the matter are evident.\n- Multiple complaints (e.g., 13091395, 13131123, 13294032, etc.) show delays, l

In [42]:
ensemble_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans mainly due to a combination of factors such as lack of proper information, mismanagement, escalating interest, and limited or misleading repayment options. Many borrowers were not fully informed about the long-term impact of interest and forbearance, or were steered into unmanageable forbearances and long-term deferments that caused their debt to grow. Others faced administrative issues like unnotified transfers of their loans, inaccurate or inconsistent information about their balances, or difficulties in establishing or maintaining manageable payment plans. Additionally, some borrowers experienced systemic misconduct by loan servicers, including errors in reporting, improper handling of their accounts, or failure to follow legal guidelines, all of which contributed to their inability to effectively pay back their loans.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [43]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [44]:
semantic_documents = semantic_chunker.split_documents(loan_complaint_data[:20])

Let's create a new vector store.

In [45]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Loan_Complaint_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [46]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [47]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [48]:
semantic_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the complaints provided, appears to be problems related to the handling and servicing of student loans. Specific issues include:\n\n- Struggling to repay or problems with repayment plans (e.g., incorrect payment amounts, lack of transparency about loan status).\n- Problems with loan reporting and credit reporting inaccuracies or illegal reporting.\n- Unauthorized or improper access to personal information and privacy violations.\n- Difficulties in communication with lenders or servicers, including long wait times and lack of clear information.\n- Disputes over loan status, defaults, or account errors.\n- Issues with loan forgiveness, cancellation, or discharge processes.\n\nOverall, the most recurring theme is mismanagement or errors in loan servicing and communication issues, leading to borrower confusion, distress, and potential violations of legal protections.'

In [49]:
semantic_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, all entries indicate that responses from the companies were marked as "Closed with explanation" and responses were reported as "Timely response? Yes." This suggests that the complaints received responses within the expected timeframe. \n\nHowever, the fact that some complaints mention ongoing issues, unresolved disputes, or repeated violations despite company responses implies that while responses were timely, the underlying issues may not have been fully handled to the complainants\' satisfaction. \n\nIn summary:  \nYes, some complaints may have not been effectively resolved or fully handled to the complainants\' satisfaction, but according to the records, the complaints were responded to in a timely manner.'

In [50]:
semantic_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans due to various issues such as miscommunication and lack of transparency from loan servicers, technical or administrative errors, difficulties in providing required documentation, improper handling of payment processing, and legal disputes over the legitimacy or status of their loans. Some borrowers also experienced challenges related to information being improperly reported or continued collection efforts after debts became legally unverified or voided. Additionally, delays or stall tactics by loan servicers appeared to contribute to borrowers' struggles in resolving their repayment issues."

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [ ]:
### YOUR CODE HERE
# import ragas llms from 7 
#uv add ragas 
#nltk?
from raqas.llms import Langchainllmwrapper 

# g




generate golde dataest
naive, bm25, reranking, etc.
should find all the code from 8
